# Making A Reward Model From NetHack Data

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

import nle.dataset as nld
from nle.nethack import tty_render
from nle.env.tasks import NetHackChallenge
from nle.language_wrapper.wrappers import nle_language_wrapper as language_wrapper

CUDA_VISIBLE_DEVICES = 0
device = torch.device(f'cuda:{CUDA_VISIBLE_DEVICES}' if torch.cuda.is_available() else 'cpu')

/homes/53/fpinto/BALROG/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Dataset prep
Need to separate dataset into separate game trajectories, and return the state, action, current reward and reward-to-go (RTG) at each step

24x80 characters in one screen

In [ ]:
dic = language_wrapper.NLELanguageWrapper.all_nle_action_map

actions = list(dic.keys())
act_str = list(dic.values())

print(len(dic))

print([i.value for i in list(language_wrapper.NLELanguageWrapper.all_nle_action_map.keys())])

for i in range(len(actions)):
    print(actions[i].name, actions[i].value, act_str[i])

126
[63, 16, 107, 108, 106, 104, 117, 110, 98, 121, 75, 76, 74, 72, 85, 78, 66, 89, 60, 62, 46, 13, 35, 191, 225, 193, 97, 24, 64, 67, 90, 227, 99, 195, 228, 100, 68, 101, 27, 69, 229, 102, 70, 230, 59, 86, 105, 73, 233, 234, 4, 92, 96, 58, 236, 237, 109, 77, 239, 111, 79, 15, 112, 44, 240, 80, 113, 241, 81, 114, 18, 82, 210, 242, 103, 71, 83, 115, 42, 34, 91, 36, 61, 43, 40, 94, 41, 33, 243, 120, 84, 65, 20, 116, 212, 95, 244, 88, 245, 246, 118, 87, 38, 47, 119, 247, 122, 45, 32, 39, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 5, 7, 9, 22, 6, 23]
HELP 63 ['help', '?']
PREVMSG 16 ['previous message', '^p']
N 107 ['north', 'k']
E 108 ['east', 'l']
S 106 ['south', 'j']
W 104 ['west', 'h']
NE 117 ['northeast', 'u']
SE 110 ['southeast', 'n']
SW 98 ['southwest', 'b']
NW 121 ['northwest', 'y']
N 75 ['far north', 'K']
E 76 ['far east', 'L']
S 74 ['far south', 'J']
W 72 ['far west', 'H']
NE 85 ['far northeast', 'U']
SE 78 ['far southeast', 'N']
SW 66 ['far southwest', 'B']
NW 89 ['far northwest', 

In [ ]:
path_to_nld_aa_taster = "./nld-aa-taster/nle_data"
dbfilename = "ttyrecs.db" 
db_conn = nld.db.connect(filename=dbfilename)

dataset = nld.TtyrecDataset(
    "taster-dataset",
    batch_size=1,
    seq_length=128,
    dbfilename=dbfilename,
) # First batch will give timesteps 0-128 of batch_size games and the second batch will provide timesteps 129-256 for the same games, etc.
minibatch = next(iter(dataset))

In [6]:
print(minibatch.keys())
print([i for i in minibatch['keypresses']])

dict_keys(['tty_chars', 'tty_colors', 'tty_cursor', 'timestamps', 'done', 'gameids', 'keypresses', 'scores'])
[array([ 27,  27,  24,  32,  32, 229,  32,  64,  92,  32,  32,  58,  47,
        77,  32,  35, 116, 101,  13,  98,  27,  27,  27,  35, 116, 101,
        13,  98,  27,  92,  32,  32,  58,  47,  77,  32, 115, 121, 115,
        98, 115, 121, 115, 121, 115,  98, 115,  98, 115, 121, 115,  98,
       115, 104, 115, 104, 115,  68,  97,  13,  99,  13,  58,  32,  44,
        32,  58,  32,  44,  32,  44,  97,  13,  58, 121, 115, 104, 115,
       107, 115, 107, 115,  98, 115, 104, 115, 110, 115, 104, 115, 104,
       115,  98, 115, 107, 115, 107, 115, 106,  98, 115, 108, 117, 104,
        98, 104, 115, 104, 115, 121, 115, 104, 115, 104, 115,  35, 116,
       101,  13,  98,  27,  98, 115, 121, 115,  58,  44,  58], dtype=uint8)]


In [ ]:
for i in minibatch['keypresses']: # Tried to decode the keypresses for the first 128 steps of a real NetHack game
    result = []
    for j in i:
        result.append(dic.get(j, []))
    print(result)

[['esc', '^['], ['esc', '^['], ['attributes', '^x'], ['space', ' '], ['space', ' '], ['enhance', 'M-e'], ['space', ' '], ['autopickup', '@'], ['known', '\\'], ['space', ' '], ['space', ' '], ['look', ':'], ['whatis', '/'], ['movefar', 'M'], ['space', ' '], ['extcmd', '#'], ['throw', 't'], ['eat', 'e'], ['more', '\r', '\\r'], ['southwest', 'b'], ['esc', '^['], ['esc', '^['], ['esc', '^['], ['extcmd', '#'], ['throw', 't'], ['eat', 'e'], ['more', '\r', '\\r'], ['southwest', 'b'], ['esc', '^['], ['known', '\\'], ['space', ' '], ['space', ' '], ['look', ':'], ['whatis', '/'], ['movefar', 'M'], ['space', ' '], ['search', 's'], ['northwest', 'y'], ['search', 's'], ['southwest', 'b'], ['search', 's'], ['northwest', 'y'], ['search', 's'], ['northwest', 'y'], ['search', 's'], ['southwest', 'b'], ['search', 's'], ['southwest', 'b'], ['search', 's'], ['northwest', 'y'], ['search', 's'], ['southwest', 'b'], ['search', 's'], ['west', 'h'], ['search', 's'], ['west', 'h'], ['search', 's'], ['droptype'

In [43]:
print(type(minibatch['scores']))
print(torch.from_numpy(minibatch['keypresses']).long().shape)
print(torch.randint(0, action_dim, (batch_size, seq_len)).shape)
minibatch['scores'].shape

<class 'numpy.ndarray'>
torch.Size([1, 128])
torch.Size([1, 128])


(1, 128)

In [22]:
print(type(minibatch['keypresses']))
minibatch['keypresses'].shape

<class 'numpy.ndarray'>


(1, 128)

In [18]:
print(type(minibatch['tty_chars']))
print(minibatch['tty_chars'].shape)
print(minibatch['tty_colors'].shape)
print(minibatch['tty_cursor'].shape)

<class 'numpy.ndarray'>
(1, 128, 24, 80)
(1, 128, 24, 80)
(1, 128, 2)


# Reward Model

In [16]:
class DecisionTransformer(nn.Module):
    def __init__(self, hidden_dim, max_len=128):
        super(DecisionTransformer, self).__init__()

        self.hidden_dim = hidden_dim
        self.max_len = max_len

        # CNN for state encoding
        self.state_embedding = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Flatten(),  # Flatten before passing to linear layer
            nn.Linear(64 * 24 * 80, hidden_dim)  # Map to hidden_dim
        )
        
        encoded_actions = [i.value for i in list(language_wrapper.NLELanguageWrapper.all_nle_action_map.keys())]

        self.action_to_idx = {action: idx for idx, action in enumerate(sorted(set(encoded_actions)))}
        self.idx_to_action = {idx: action for action, idx in self.action_to_idx.items()}

        # Action embedding layer with fixed size 126
        self.action_embedding = nn.Embedding(len(self.action_to_idx), hidden_dim)

        self.projection = nn.Linear(hidden_dim * 2, hidden_dim)

        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=hidden_dim, nhead=8), 
            num_layers=6
        )

        self.score_predictor = nn.Linear(hidden_dim, 1)


    def forward(self, state, action):
        batch_size, seq_length, height, width = state.shape

        # Reshape for CNN: [batch_size * seq_len, 1, 24, 80]
        state = state.view(-1, 1, height, width)
        state_embedded = self.state_embedding(state)  # [batch_size * seq_len, hidden_dim]

        state_embedded = state_embedded.view(batch_size, seq_length, self.hidden_dim) # Reshape back to [batch_size, seq_len, hidden_dim]

        action_remapped = torch.tensor([[self.action_to_idx.get(a.item(), 0) for a in seq] for seq in action])
        action_embedded = self.action_embedding(action_remapped.to(action.device))

        
        x = torch.cat((state_embedded, action_embedded), dim=-1) # Concatenate state and action embeddings
        x = self.projection(x)

        transformer_out = self.transformer(x)
        score_preds = self.score_predictor(transformer_out)

        return score_preds.squeeze(-1)  # [batch_size, seq_len]

In [18]:
# Dummy data (for illustration)
batch_size = 1
seq_len = 128
hidden_dim = 64

# Data
states = torch.from_numpy(minibatch['tty_chars']).float()
actions = torch.from_numpy(minibatch['keypresses']).long() # Will have to remap these
scores = torch.from_numpy(minibatch['scores']).float()

# Initialize model
model = DecisionTransformer(hidden_dim)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Training loop
model.train()
optimizer.zero_grad()

# Forward pass
predicted_scores = model(states, actions)

# Loss calculation
loss = criterion(predicted_scores, scores)

# Backpropagation
loss.backward()
optimizer.step()

print(f"Predicted scores: {predicted_scores}")
print(f"Loss: {loss.item():.4f}")

Predicted scores: tensor([[-0.8602, -0.7910, -1.3074, -1.0764, -0.8283, -1.0814, -1.1009, -0.6358,
         -1.1862, -0.8209, -0.9145, -0.5644, -1.3646, -1.2049, -1.1275, -0.8699,
         -0.3016, -1.2747, -1.2128, -0.8437, -1.3899, -0.8594, -0.6118, -1.2837,
         -0.8487, -1.1108, -0.9640, -0.8438, -1.0549, -1.1340, -1.0513, -0.7869,
         -0.7957, -0.7070, -1.4179, -1.5419, -1.0498, -0.5772, -0.7579, -0.6795,
         -1.2356, -1.2704, -0.9144, -1.1838, -0.8155, -1.0745, -1.1000, -0.6832,
         -1.2926, -0.9071, -0.8600, -1.0213, -1.1083, -0.9673, -0.8671, -0.8430,
         -1.1770, -0.9071, -1.0194, -0.6324, -1.0444, -1.1379, -0.3312, -0.7746,
         -0.7948, -0.8984, -0.1816, -0.8950, -1.4396, -1.1637, -0.9951, -1.0200,
         -1.1621, -0.5667, -1.2606, -0.6510, -1.1394, -1.3849, -1.5278, -1.1488,
         -0.8704, -0.8297, -0.9628, -1.2995, -0.9284, -1.5200, -0.9280, -1.2430,
         -0.2585, -0.6416, -0.7469, -1.1841, -1.2786, -1.1924, -0.8114, -0.8262,
         -